In [1]:
import pandas as pd

studies = pd.read_csv('/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/studies.txt', sep='|')
conditions = pd.read_csv('/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/conditions.txt', sep='|')

# Filter conditions for PDAC
pdac_terms = ['pancreatic cancer', 'pancreatic ductal', 
              'pancreatic adenocarcinoma', 'PDAC', 'pancreatic neoplasm']

pdac_nctids = conditions[
    conditions['name'].str.lower().str.contains('pancrea', na=False)
]['nct_id'].unique()

pdac_studies = studies[studies['nct_id'].isin(pdac_nctids)]

/var/folders/p6/ycjyqz1s44s4gfbdb_gdpxcw0000gr/T/ipykernel_41765/2242812887.py:3: DtypeWarning: Columns (46,47,48,53,68) have mixed types. Specify dtype option on import or set low_memory=False.
  studies = pd.read_csv('/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/studies.txt', sep='|')


In [2]:
pdac_studies.head()

,nct_id,nlm_download_date_description,study_first_submitted_date,results_first_submitted_date,disposition_first_submitted_date,last_update_submitted_date,study_first_submitted_qc_date,study_first_posted_date,study_first_posted_date_type,results_first_submitted_qc_date,...,plan_to_share_ipd_description,created_at,updated_at,source_class,delayed_posting,expanded_access_nctid,expanded_access_status_for_nctid,fdaaa801_violation,baseline_type_units_analyzed,patient_registry
114,NCT06277414,NaN,2024-02-19,NaN,NaN,2024-02-19,2024-02-19,2024-02-26,ACTUAL,NaN,...,NaN,2026-04-11 19:30:46.942396,2026-04-11 19:30:46.942396,OTHER,NaN,NaN,NaN,NaN,NaN,t
160,NCT03201926,NaN,2017-06-22,NaN,NaN,2019-01-15,2017-06-26,2017-06-28,ACTUAL,NaN,...,NaN,2026-04-11 19:30:46.942396,2026-04-11 19:30:46.942396,OTHER,NaN,NaN,NaN,NaN,NaN,NaN
293,NCT04901949,NaN,2021-05-03,NaN,NaN,2021-05-20,2021-05-20,2021-05-26,ACTUAL,NaN,...,NaN,2026-04-11 19:30:46.942396,2026-04-11 19:30:46.942396,OTHER,NaN,NaN,NaN,NaN,NaN,f
321,NCT06131840,NaN,2023-11-09,NaN,NaN,2026-04-06,2023-11-09,2023-11-14,ACTUAL,NaN,...,Pfizer will provide access to individual de-id...,2026-04-12 04:05:04.982706,2026-04-12 04:05:04.982706,INDUSTRY,NaN,NaN,NaN,NaN,NaN,NaN
488,NCT04449406,NaN,2020-05-05,NaN,NaN,2026-04-01,2020-06-25,2020-06-26,ACTUAL,NaN,...,NaN,2026-04-12 04:05:04.982706,2026-04-12 04:05:04.982706,OTHER,NaN,NaN,NaN,NaN,NaN,f


In [3]:
# How many unique trials
print(f"Total PDAC trials: {len(pdac_studies)}")

# What columns exist
print(pdac_studies.columns.tolist())

# What study phases exist and how many
print(pdac_studies['phase'].value_counts(dropna=False))

# What overall statuses exist
print(pdac_studies['overall_status'].value_counts(dropna=False))

# What study types exist
print(pdac_studies['study_type'].value_counts(dropna=False))

Total PDAC trials: 5948
['nct_id', 'nlm_download_date_description', 'study_first_submitted_date', 'results_first_submitted_date', 'disposition_first_submitted_date', 'last_update_submitted_date', 'study_first_submitted_qc_date', 'study_first_posted_date', 'study_first_posted_date_type', 'results_first_submitted_qc_date', 'results_first_posted_date', 'results_first_posted_date_type', 'disposition_first_submitted_qc_date', 'disposition_first_posted_date', 'disposition_first_posted_date_type', 'last_update_submitted_qc_date', 'last_update_posted_date', 'last_update_posted_date_type', 'start_month_year', 'start_date_type', 'start_date', 'verification_month_year', 'verification_date', 'completion_month_year', 'completion_date_type', 'completion_date', 'primary_completion_month_year', 'primary_completion_date_type', 'primary_completion_date', 'target_duration', 'study_type', 'acronym', 'baseline_population', 'brief_title', 'official_title', 'overall_status', 'last_known_status', 'phase', 'en

In [4]:
# Step 1: Keep only interventional studies
# Why: only these test a treatment and can have a success/failure outcome
df = pdac_studies[pdac_studies['study_type'] == 'INTERVENTIONAL'].copy()
print(f"After interventional filter: {len(df)}")

# Step 2: Keep only relevant phases
# Why: Phase 1 = safety only (no efficacy endpoint to predict)
#      Phase 4 = post-approval (different question entirely)
#      Early Phase 1 = even more preliminary than Phase 1
#      We keep Phase 2, Phase 3, Phase 1/2, Phase 2/3
keep_phases = ['PHASE2', 'PHASE3', 'PHASE1/PHASE2', 'PHASE2/PHASE3']
df = df[df['phase'].isin(keep_phases)].copy()
print(f"After phase filter: {len(df)}")

# Step 3: Keep only completed or terminated trials
# Why: these are finished — they either have results or a documented reason for stopping
#      Recruiting/active trials have no outcome yet, we can't label them
keep_status = ['COMPLETED', 'TERMINATED']
df = df[df['overall_status'].isin(keep_status)].copy()
print(f"After status filter: {len(df)}")

# Step 4: Look at what we have
print(df['phase'].value_counts())
print(df['overall_status'].value_counts())

After interventional filter: 4522
After phase filter: 2060
After status filter: 1090
phase
PHASE2           660
PHASE1/PHASE2    228
PHASE3           174
PHASE2/PHASE3     28
Name: count, dtype: int64
overall_status
COMPLETED     831
TERMINATED    259
Name: count, dtype: int64


In [5]:
df.head()

,nct_id,nlm_download_date_description,study_first_submitted_date,results_first_submitted_date,disposition_first_submitted_date,last_update_submitted_date,study_first_submitted_qc_date,study_first_posted_date,study_first_posted_date_type,results_first_submitted_qc_date,...,plan_to_share_ipd_description,created_at,updated_at,source_class,delayed_posting,expanded_access_nctid,expanded_access_status_for_nctid,fdaaa801_violation,baseline_type_units_analyzed,patient_registry
746,NCT04426669,NaN,2020-06-08,NaN,NaN,2026-02-19,2020-06-09,2020-06-11,ACTUAL,NaN,...,NaN,2026-04-11 19:30:46.942396,2026-04-11 19:30:46.942396,INDUSTRY,NaN,NaN,NaN,NaN,NaN,NaN
1109,NCT02653313,NaN,2015-12-04,NaN,NaN,2022-11-16,2016-01-08,2016-01-12,ESTIMATED,NaN,...,NaN,2026-04-12 23:42:59.612378,2026-04-12 23:42:59.612378,INDUSTRY,NaN,NaN,NaN,NaN,NaN,NaN
1801,NCT00655785,NaN,2008-04-04,NaN,NaN,2013-03-13,2008-04-04,2008-04-10,ESTIMATED,NaN,...,NaN,2026-04-12 14:08:29.834389,2026-04-12 14:08:29.834389,OTHER,NaN,NaN,NaN,NaN,NaN,NaN
2530,NCT00041639,NaN,2002-07-11,NaN,NaN,2021-08-12,2002-07-12,2002-07-15,ESTIMATED,NaN,...,NaN,2026-04-12 14:10:42.228337,2026-04-12 14:10:42.228337,INDUSTRY,NaN,NaN,NaN,NaN,NaN,NaN
2586,NCT02017015,NaN,2013-12-05,2016-04-06,NaN,2017-08-28,2013-12-16,2013-12-20,ESTIMATED,2016-10-03,...,NaN,2026-04-11 11:18:11.013955,2026-04-11 11:18:11.013955,INDUSTRY,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
# Load the relevant tables
outcomes = pd.read_csv('/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/outcomes.txt', sep='|')
outcome_measurements = pd.read_csv('/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/outcome_measurements.txt', sep='|')
outcome_analyses = pd.read_csv('/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/outcome_analyses.txt', sep='|')

# Quick look at structure
print("=== OUTCOMES ===")
print(outcomes.columns.tolist())
print(outcomes.head(3))

print("\n=== OUTCOME_MEASUREMENTS ===")
print(outcome_measurements.columns.tolist())
print(outcome_measurements.head(3))

print("\n=== OUTCOME_ANALYSES ===")
print(outcome_analyses.columns.tolist())
print(outcome_analyses.head(3))

# Now check coverage for our 1090 trials
our_nctids = set(df['nct_id'])

trials_with_outcomes = outcomes[outcomes['nct_id'].isin(our_nctids)]['nct_id'].nunique()
trials_with_measurements = outcome_measurements[outcome_measurements['nct_id'].isin(our_nctids)]['nct_id'].nunique()
trials_with_analyses = outcome_analyses[outcome_analyses['nct_id'].isin(our_nctids)]['nct_id'].nunique()

print(f"\nTrials with outcomes defined: {trials_with_outcomes} / {len(df)}")
print(f"Trials with measurements: {trials_with_measurements} / {len(df)}")
print(f"Trials with statistical analyses: {trials_with_analyses} / {len(df)}")

/var/folders/p6/ycjyqz1s44s4gfbdb_gdpxcw0000gr/T/ipykernel_41765/752358453.py:3: DtypeWarning: Columns (11,14,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  outcome_measurements = pd.read_csv('/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/outcome_measurements.txt', sep='|')
/var/folders/p6/ycjyqz1s44s4gfbdb_gdpxcw0000gr/T/ipykernel_41765/752358453.py:4: DtypeWarning: Columns (22,23) have mixed types. Specify dtype option on import or set low_memory=False.
  outcome_analyses = pd.read_csv('/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/outcome_analyses.txt', sep='|')


=== OUTCOMES ===
['id', 'nct_id', 'outcome_type', 'title', 'description', 'time_frame', 'population', 'anticipated_posting_date', 'anticipated_posting_month_year', 'units', 'units_analyzed', 'dispersion_type', 'param_type']
          id       nct_id outcome_type  \
0  211630855  NCT01256476      PRIMARY   
1  211630856  NCT01796665      PRIMARY   
2  211630857  NCT01796665      PRIMARY   

                                               title              description  \
0  Mean Percent Change in Low Density Lipoprotein...                      NaN   
1  Mean Percent Change in the Inflammatory (Papul...  Per protocol population   
2  Mean Percent Change in the Non-inflammatory (O...                      NaN   

              time_frame                                         population  \
0  Baseline and 12 weeks  All randomized subjects who took at least 1 do...   
1    Baseline to week 12                            Per protocol population   
2    Baseline to week 12                     

In [7]:
# Load study references — this links trials to their publications
study_references = pd.read_csv('/Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/study_references.txt', sep='|')

print(study_references.columns.tolist())
print(study_references.head(3))

# How many of our 1090 trials have a publication linked?
our_refs = study_references[study_references['nct_id'].isin(our_nctids)]
trials_with_refs = our_refs['nct_id'].nunique()
print(f"\nTrials with publication links: {trials_with_refs} / {len(df)}")

# What types of references exist?
print(our_refs['reference_type'].value_counts(dropna=False))

# How many have a PMID (PubMed ID) — these we can fetch automatically
if 'pmid' in our_refs.columns:
    has_pmid = our_refs[our_refs['pmid'].notna()]['nct_id'].nunique()
    print(f"Trials with a PMID: {has_pmid}")

['id', 'nct_id', 'pmid', 'reference_type', 'citation']
          id       nct_id        pmid reference_type  \
0  350655945  NCT01702454  30325891.0        DERIVED   
1  350655946  NCT04981743  31986264.0     BACKGROUND   
2  350655947  NCT04981743  32179124.0     BACKGROUND   

                                            citation  
0  Claeys C, Chandrasekaran V, Garcia-Sicilia J, ...  
1  Huang C, Wang Y, Li X, Ren L, Zhao J, Hu Y, Zh...  
2  Rodriguez-Morales AJ, Cardona-Ospina JA, Gutie...  

Trials with publication links: 470 / 1090
reference_type
BACKGROUND    755
DERIVED       423
RESULT        194
Name: count, dtype: int64
Trials with a PMID: 463


In [8]:
# Get only RESULT references with a PMID for our trials
result_refs = our_refs[
    (our_refs['reference_type'] == 'RESULT') & 
    (our_refs['pmid'].notna())
].copy()

print(f"RESULT references with PMID: {len(result_refs)}")
print(f"Unique trials covered: {result_refs['nct_id'].nunique()}")

# Clean PMID — make sure it's an integer string
result_refs['pmid'] = result_refs['pmid'].astype(int).astype(str)

print(result_refs[['nct_id', 'pmid', 'citation']].head(10))

RESULT references with PMID: 151
Unique trials covered: 95
             nct_id      pmid  \
4006    NCT01195415  25278454   
15296   NCT00003018  17470859   
25186   NCT00003546  15361643   
45068   NCT02558868  24982456   
55383   NCT01523457  27022826   
69598   NCT02879318  36028483   
74592   NCT01781520  28611200   
92513   NCT00100815  19491904   
118357  NCT01280058  27039845   
119268  NCT03340974  33427657   

                                                 citation  
4006    Kim EJ, Sahai V, Abel EV, Griffith KA, Greenso...  
15296   Isacoff WH, Bendetti JK, Barstis JJ, Jazieh AR...  
25186   Blackstock AW, Tepper JE, Niedwiecki D, Hollis...  
45068   Oettle H, Riess H, Stieler JM, Heil G, Schwane...  
55383   Stein SM, James ES, Deng Y, Cong X, Kortmansky...  
69598   Renouf DJ, Loree JM, Knox JJ, Topham JT, Kavan...  
74592   Jiang N, Qiao G, Wang X, Morse MA, Gwin WR, Zh...  
92513   Javle M, Yu J, Garrett C, Pande A, Kuvshinoff ...  
118357  Noonan AM, Farren MR, Geyer S

In [9]:
from Bio import Entrez
import time

# You must provide an email — PubMed requires it to contact you if you abuse the API
# It's not stored anywhere sensitive, just put yours
Entrez.email = "ammigiuli@gmail.com"

def fetch_abstract(pmid):
    """Fetch abstract text for a single PMID from PubMed."""
    try:
        handle = Entrez.efetch(
            db="pubmed",
            id=str(pmid),
            rettype="abstract",
            retmode="xml"
        )
        record = Entrez.read(handle)
        handle.close()
        
        # Navigate the XML structure to get abstract text
        article = record['PubmedArticle'][0]['MedlineCitation']['Article']
        
        if 'Abstract' in article:
            abstract_text = ' '.join(article['Abstract']['AbstractText'])
            title = article['ArticleTitle']
            return {'pmid': pmid, 'title': str(title), 'abstract': str(abstract_text)}
        else:
            return {'pmid': pmid, 'title': str(article.get('ArticleTitle', '')), 'abstract': ''}
    
    except Exception as e:
        print(f"Failed for PMID {pmid}: {e}")
        return {'pmid': pmid, 'title': '', 'abstract': ''}

# Fetch all 151 abstracts
# We pause 0.34 seconds between requests — PubMed allows max 3 requests/second
abstracts = []
pmid_list = result_refs['pmid'].tolist()

for i, pmid in enumerate(pmid_list):
    result = fetch_abstract(pmid)
    abstracts.append(result)
    
    if (i + 1) % 10 == 0:
        print(f"Fetched {i+1}/{len(pmid_list)}")
    
    time.sleep(0.34)

# Convert to DataFrame
abstracts_df = pd.DataFrame(abstracts)

# Merge with the nct_id so we know which trial each abstract belongs to
abstracts_df = abstracts_df.merge(
    result_refs[['nct_id', 'pmid']], 
    on='pmid', 
    how='left'
)

print(f"\nTotal fetched: {len(abstracts_df)}")
print(f"With non-empty abstract: {(abstracts_df['abstract'] != '').sum()}")

# Save immediately — don't lose this
abstracts_df.to_csv('pdac_trial_abstracts.csv', index=False)
print("Saved to pdac_trial_abstracts.csv")

Fetched 10/151
Fetched 20/151
Fetched 30/151
Fetched 40/151
Fetched 50/151
Fetched 60/151
Fetched 70/151
Fetched 80/151
Fetched 90/151
Fetched 100/151
Fetched 110/151
Fetched 120/151
Fetched 130/151
Fetched 140/151
Fetched 150/151

Total fetched: 151
With non-empty abstract: 142
Saved to pdac_trial_abstracts.csv


In [16]:
def classify_abstract(abstract, title=""):
    
    text = (abstract + " " + title).lower()
    
    # --- EXCLUDE: feasibility/pilot trials ---
    # These have no efficacy primary endpoint
    feasibility_patterns = [
        r"feasibility (study|trial|of)",
        r"pilot (study|trial)",
        r"confirm.{0,30}feasibility",
        r"primary.{0,50}(feasibility|safety|tolerability|dose)",
        r"phase i\b",  # pure phase I
    ]
    is_feasibility = any(re.search(p, text) for p in feasibility_patterns)
    if is_feasibility:
        return -2, 0, 0  # -2 = exclude entirely

    # --- FAILURE signals ---
    failure_patterns = [
        r"did not meet.{0,50}primary endpoint",
        r"failed to (meet|demonstrate|show|achieve)",
        r"no significant (difference|improvement|benefit)",
        r"did not (significantly|substantially) improve",
        r"not (significantly|statistically) different",
        r"terminated.{0,50}(futility|lack of efficacy|inefficacy)",
        r"(primary endpoint|primary outcome).{0,80}not (met|reached|achieved)",
        r"negative (trial|study|result)",
        r"did not (reach|achieve).{0,50}(primary|endpoint|significance)",
        r"no (overall survival|progression.free survival).{0,50}benefit",
        r"futility",
        r"p\s*[=\s>]\s*0\.[1-9]",   # p > 0.1 — clearly not significant
        r"p\s*=\s*0\.(?:0[6-9]|[1-9])",  # p = 0.06 to 0.99
    ]
    
    # --- SUCCESS signals ---
    success_patterns = [
        r"(met|achieved|reached).{0,50}primary (endpoint|outcome)",
        r"(primary endpoint|primary outcome).{0,80}(met|achieved|reached|significant)",
        r"significantly (improved|prolonged|extended|increased)",
        r"superior (to|over).{0,50}(placebo|control|standard)",
        r"statistically significant (improvement|increase|reduction|benefit)",
        r"(demonstrated|showed).{0,50}significant(ly)?.{0,30}(improvement|benefit|efficacy)",
        r"improved (overall survival|progression.free survival)",
        r"(overall survival|progression.free survival).{0,50}(significant|improved|prolonged)",
        r"p\s*[=<]\s*0\.0[0-4]\d*",  # p < 0.05
        r"p\s*<\s*0\.05",
        r"hazard ratio.{0,50}p\s*[=<]\s*0\.0[0-4]",  # HR with significant p
    ]
    
    failure_score = sum(1 for p in failure_patterns if re.search(p, text))
    success_score = sum(1 for p in success_patterns if re.search(p, text))
    
    if failure_score > 0 and success_score == 0:
        return 0, failure_score, success_score
    elif success_score > 0 and failure_score == 0:
        return 1, failure_score, success_score
    elif success_score > 0 and failure_score > 0:
        return -1, failure_score, success_score
    else:
        return -1, failure_score, success_score

# Re-apply
abstracts_df['label'], abstracts_df['failure_score'], abstracts_df['success_score'] = zip(
    *abstracts_df['abstract'].apply(lambda x: classify_abstract(x))
)

print("Label distribution:")
print(f"Success  (1):  {(abstracts_df['label']==1).sum()}")
print(f"Failure  (0):  {(abstracts_df['label']==0).sum()}")
print(f"Uncertain(-1): {(abstracts_df['label']==-1).sum()}")
print(f"Excluded (-2): {(abstracts_df['label']==-2).sum()}")

Label distribution:
Success  (1):  27
Failure  (0):  6
Uncertain(-1): 98
Excluded (-2): 20


In [20]:
import requests
import json
import time

def classify_with_mistral(abstract, title, nct_id):
    """
    Use local Mistral via Ollama to classify trial outcome.
    """
    
    prompt = f"""You are an expert in clinical oncology reading a clinical trial abstract.
Your task is to determine whether this trial met its PRIMARY endpoint.

Trial: {nct_id}
Title: {title}
Abstract: {abstract}

Classify this trial as exactly one of:
- SUCCESS: The trial met its pre-specified primary endpoint (statistically significant result)
- FAILURE: The trial did NOT meet its primary endpoint (null result, futility, no significant difference)
- EXCLUDE: Not an interventional efficacy trial (feasibility, biomarker, QoL only, Phase I dose-finding, pooled analysis)
- UNCERTAIN: Abstract genuinely does not contain enough information

Respond ONLY with valid JSON, no other text, no markdown:
{{"label": "SUCCESS" or "FAILURE" or "EXCLUDE" or "UNCERTAIN", "confidence": "HIGH" or "MEDIUM" or "LOW", "primary_endpoint": "brief description of primary endpoint", "reasoning": "one sentence explanation"}}"""

    try:
        response = requests.post(
            'http://localhost:11434/api/generate',
            json={
                'model': 'mistral',
                'prompt': prompt,
                'stream': False,
                'options': {
                    'temperature': 0.1,  # low temperature = more consistent, less creative
                }
            },
            timeout=60
        )
        
        raw = response.json()['response'].strip()
        
        # Clean up in case model adds markdown fences
        raw = raw.replace('```json', '').replace('```', '').strip()
        
        result = json.loads(raw)
        result['nct_id'] = nct_id
        return result
    
    except json.JSONDecodeError:
        # Model didn't return valid JSON — save raw response for inspection
        print(f"JSON parse failed for {nct_id}. Raw: {raw[:200]}")
        return {
            'nct_id': nct_id,
            'label': 'UNCERTAIN',
            'confidence': 'LOW',
            'primary_endpoint': '',
            'reasoning': 'JSON parse error'
        }
    except Exception as e:
        print(f"Failed for {nct_id}: {e}")
        return {
            'nct_id': nct_id,
            'label': 'UNCERTAIN',
            'confidence': 'LOW',
            'primary_endpoint': '',
            'reasoning': f'Error: {str(e)}'
        }

# --- Test on a single abstract first ---
test_row = uncertain.iloc[0]
test_result = classify_with_mistral(
    abstract=test_row['abstract'],
    title=test_row['title'],
    nct_id=test_row['nct_id']
)
print("Test result:")
print(json.dumps(test_result, indent=2))

Test result:
{
  "label": "FAILURE",
  "confidence": "HIGH",
  "primary_endpoint": "Evaluating the efficacy of GDC-0449 and gemcitabine in treating metastatic pancreatic cancer compared to gemcitabine alone",
  "reasoning": "The trial did not show that the combination therapy was superior to gemcitabine alone, as there was no significant difference in response and disease control rate, progression-free survival, or overall survival.",
  "nct_id": "NCT01195415"
}


In [21]:
# Run on all 98 uncertain abstracts
claude_labels = []

for i, row in uncertain.iterrows():
    result = classify_with_mistral(
        abstract=row['abstract'],
        title=row['title'],
        nct_id=row['nct_id']
    )
    claude_labels.append(result)
    
    if (len(claude_labels)) % 10 == 0:
        print(f"Progress: {len(claude_labels)}/98")
    
    time.sleep(0.2)

# Convert to DataFrame
mistral_df = pd.DataFrame(claude_labels)

print("\n=== Mistral classification results ===")
print(mistral_df['label'].value_counts())
print(f"\nHigh confidence:   {(mistral_df['confidence']=='HIGH').sum()}")
print(f"Medium confidence: {(mistral_df['confidence']=='MEDIUM').sum()}")
print(f"Low confidence:    {(mistral_df['confidence']=='LOW').sum()}")

# Save
mistral_df.to_csv('pdac_mistral_labels.csv', index=False)

Progress: 10/98
Progress: 20/98
Progress: 30/98
Progress: 40/98
Progress: 50/98
Progress: 60/98
Progress: 70/98
Progress: 80/98
Progress: 90/98
Progress: 100/98
Progress: 110/98
Progress: 120/98
Progress: 130/98

=== Mistral classification results ===
label
SUCCESS      66
FAILURE      42
UNCERTAIN    22
EXCLUDE       1
Name: count, dtype: int64

High confidence:   109
Medium confidence: 0
Low confidence:    22


In [22]:
# === BUILD MASTER LABELED DATASET ===

# 1. From regex classifier — the ones already labeled with confidence
regex_labeled = abstracts_df[abstracts_df['label'].isin([0, 1])][['nct_id', 'pmid', 'title', 'abstract', 'label']].copy()
regex_labeled['source'] = 'regex'
regex_labeled['confidence'] = 'HIGH'
# Convert 0/1 to string labels to match mistral format
regex_labeled['label'] = regex_labeled['label'].map({1: 'SUCCESS', 0: 'FAILURE'})
print(f"From regex: {len(regex_labeled)} trials")

# 2. From mistral — high confidence only (LOW confidence = unreliable)
mistral_high = mistral_df[
    (mistral_df['confidence'] == 'HIGH') &
    (mistral_df['label'].isin(['SUCCESS', 'FAILURE']))
][['nct_id', 'label', 'confidence', 'primary_endpoint', 'reasoning']].copy()
mistral_high['source'] = 'mistral'

# Merge with abstract text
mistral_high = mistral_high.merge(
    abstracts_df[['nct_id', 'pmid', 'title', 'abstract']],
    on='nct_id',
    how='left'
)
print(f"From mistral (high confidence): {len(mistral_high)} trials")

# 3. Combine
master = pd.concat([regex_labeled, mistral_high], ignore_index=True)

# 4. Remove duplicates — if a trial was labeled by both, keep regex (more conservative)
master = master.drop_duplicates(subset='nct_id', keep='first')

print(f"\nTotal labeled trials: {len(master)}")
print(f"SUCCESS: {(master['label']=='SUCCESS').sum()}")
print(f"FAILURE: {(master['label']=='FAILURE').sum()}")
print(f"Success rate: {(master['label']=='SUCCESS').sum()/len(master)*100:.1f}%")

# 5. Merge with study metadata from our filtered df
master = master.merge(
    df[['nct_id', 'phase', 'overall_status', 'enrollment', 
        'start_date', 'completion_date', 'study_type']],
    on='nct_id',
    how='left'
)

# 6. Save — this is your most important file so far
master.to_csv('pdac_master_labeled.csv', index=False)
print("\nSaved to pdac_master_labeled.csv")

# 7. Quick sanity check by phase
print("\nSuccess rate by phase:")
print(master.groupby('phase')['label'].apply(
    lambda x: f"{(x=='SUCCESS').sum()}/{len(x)} ({(x=='SUCCESS').mean()*100:.0f}%)"
))

From regex: 33 trials
From mistral (high confidence): 407 trials

Total labeled trials: 84
SUCCESS: 47
FAILURE: 37
Success rate: 56.0%

Saved to pdac_master_labeled.csv

Success rate by phase:
phase
PHASE1/PHASE2      6/8 (75%)
PHASE2           27/55 (49%)
PHASE2/PHASE3      2/3 (67%)
PHASE3           12/18 (67%)
Name: label, dtype: object


In [23]:
# First understand the duplication issue
print(f"Mistral df total rows: {len(mistral_df)}")
print(f"Mistral unique NCT IDs: {mistral_df['nct_id'].nunique()}")

# How many were classified more than once?
dup_counts = mistral_df['nct_id'].value_counts()
print(f"Trials classified once: {(dup_counts==1).sum()}")
print(f"Trials classified multiple times: {(dup_counts>1).sum()}")
print(f"Max times a trial was classified: {dup_counts.max()}")

# For duplicates — did mistral agree with itself?
duplicated_ids = dup_counts[dup_counts > 1].index
dup_df = mistral_df[mistral_df['nct_id'].isin(duplicated_ids)]
print("\nFor duplicated trials, label consistency:")
consistency = dup_df.groupby('nct_id')['label'].nunique()
print(f"Always same label: {(consistency==1).sum()}")
print(f"Disagreed with itself: {(consistency>1).sum()}")

Mistral df total rows: 131
Mistral unique NCT IDs: 81
Trials classified once: 68
Trials classified multiple times: 13
Max times a trial was classified: 12

For duplicated trials, label consistency:
Always same label: 4
Disagreed with itself: 9


In [24]:
# Fix deduplication — for each nct_id keep the most common label
# If mistral disagreed with itself, that trial goes to UNCERTAIN

from collections import Counter

def resolve_duplicates(group):
    labels = group['label'].tolist()
    label_counts = Counter(labels)
    most_common, count = label_counts.most_common(1)[0]
    
    # If there's a clear majority, keep it
    if count > len(labels) / 2:
        return group[group['label'] == most_common].iloc[0]
    else:
        # No majority — mark uncertain
        row = group.iloc[0].copy()
        row['label'] = 'UNCERTAIN'
        row['confidence'] = 'LOW'
        return row

mistral_dedup = mistral_df.groupby('nct_id', group_keys=False).apply(resolve_duplicates)
print(f"\nAfter proper deduplication: {len(mistral_dedup)} unique trials")
print(mistral_dedup['label'].value_counts())


After proper deduplication: 81 unique trials
label
FAILURE      37
SUCCESS      32
UNCERTAIN    12
Name: count, dtype: int64


/var/folders/p6/ycjyqz1s44s4gfbdb_gdpxcw0000gr/T/ipykernel_41765/3091943308.py:21: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mistral_dedup = mistral_df.groupby('nct_id', group_keys=False).apply(resolve_duplicates)


In [27]:
# Reset index after groupby to remove the ambiguity
mistral_dedup = mistral_df.groupby('nct_id', group_keys=False).apply(resolve_duplicates)
mistral_dedup = mistral_dedup.reset_index(drop=True)  # <-- add this line

print(f"\nAfter proper deduplication: {len(mistral_dedup)} unique trials")
print(mistral_dedup['label'].value_counts())

# Now continue
mistral_high_clean = mistral_dedup[
    (mistral_dedup['confidence'] == 'HIGH') &
    (mistral_dedup['label'].isin(['SUCCESS', 'FAILURE']))
][['nct_id', 'label', 'confidence', 'primary_endpoint', 'reasoning']].copy()
mistral_high_clean['source'] = 'mistral'

mistral_high_clean = mistral_high_clean.merge(
    abstracts_df[['nct_id', 'pmid', 'title', 'abstract']],
    on='nct_id',
    how='left'
)

# Rebuild master
master2 = pd.concat([regex_labeled, mistral_high_clean], ignore_index=True)
master2 = master2.drop_duplicates(subset='nct_id', keep='first')

master2 = master2.merge(
    df[['nct_id', 'phase', 'overall_status', 'enrollment',
        'start_date', 'completion_date']],
    on='nct_id', how='left'
)

print(f"\nRebuilt master: {len(master2)} trials")
print(f"SUCCESS: {(master2['label']=='SUCCESS').sum()}")
print(f"FAILURE: {(master2['label']=='FAILURE').sum()}")
print(f"Success rate: {(master2['label']=='SUCCESS').sum()/len(master2)*100:.1f}%")

print("\nBy phase:")
print(master2.groupby('phase')['label'].apply(
    lambda x: f"{(x=='SUCCESS').sum()}/{len(x)} ({(x=='SUCCESS').mean()*100:.0f}%)"
))

master2.to_csv('pdac_master_labeled_v2.csv', index=False)
print("\nSaved.")


After proper deduplication: 81 unique trials
label
FAILURE      37
SUCCESS      32
UNCERTAIN    12
Name: count, dtype: int64

Rebuilt master: 83 trials
SUCCESS: 47
FAILURE: 36
Success rate: 56.6%

By phase:
phase
PHASE1/PHASE2      6/8 (75%)
PHASE2           27/54 (50%)
PHASE2/PHASE3      2/3 (67%)
PHASE3           12/18 (67%)
Name: label, dtype: object

Saved.


/var/folders/p6/ycjyqz1s44s4gfbdb_gdpxcw0000gr/T/ipykernel_41765/2048625970.py:2: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  mistral_dedup = mistral_df.groupby('nct_id', group_keys=False).apply(resolve_duplicates)


In [30]:
# First let's see what we're still missing
# How many of our 1090 trials have NO label yet?

labeled_ids = set(master2['nct_id'])
unlabeled = df[~df['nct_id'].isin(labeled_ids)].copy()
print(f"Still unlabeled: {len(unlabeled)}")
print(f"Of which COMPLETED: {(unlabeled['overall_status']=='COMPLETED').sum()}")
print(f"Of which TERMINATED: {(unlabeled['overall_status']=='TERMINATED').sum()}")

# Terminated trials are almost always failures
# This is a clean labeling rule we can apply directly
terminated_unlabeled = unlabeled[unlabeled['overall_status'] == 'TERMINATED'].copy()
print(f"\nTerminated with no label: {len(terminated_unlabeled)}")

# Check their termination reasons
# Load the milestones or designs table for termination reasons
designs = pd.read_csv('Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/milestones.txt', sep='|')
our_terminated = designs[designs['nct_id'].isin(terminated_unlabeled['nct_id'])]
print(our_terminated.columns.tolist())

Still unlabeled: 1007
Of which COMPLETED: 755
Of which TERMINATED: 252

Terminated with no label: 252


FileNotFoundError: [Errno 2] No such file or directory: 'Users/giuliaam/Desktop/Newlife/ML_Project/Clinical_Trial/Data/milestones.txt'

In [31]:
# Check if why_stopped is in the studies table
print([col for col in pdac_studies.columns if 'stop' in col.lower() 
       or 'terminat' in col.lower() 
       or 'why' in col.lower()])

# Also check what columns studies has in general
print(pdac_studies.columns.tolist())

['why_stopped']
['nct_id', 'nlm_download_date_description', 'study_first_submitted_date', 'results_first_submitted_date', 'disposition_first_submitted_date', 'last_update_submitted_date', 'study_first_submitted_qc_date', 'study_first_posted_date', 'study_first_posted_date_type', 'results_first_submitted_qc_date', 'results_first_posted_date', 'results_first_posted_date_type', 'disposition_first_submitted_qc_date', 'disposition_first_posted_date', 'disposition_first_posted_date_type', 'last_update_submitted_qc_date', 'last_update_posted_date', 'last_update_posted_date_type', 'start_month_year', 'start_date_type', 'start_date', 'verification_month_year', 'verification_date', 'completion_month_year', 'completion_date_type', 'completion_date', 'primary_completion_month_year', 'primary_completion_date_type', 'primary_completion_date', 'target_duration', 'study_type', 'acronym', 'baseline_population', 'brief_title', 'official_title', 'overall_status', 'last_known_status', 'phase', 'enrollment

In [34]:
# Look at why_stopped for our terminated unlabeled trials
terminated_why = unlabeled[unlabeled['overall_status'] == 'TERMINATED'][['nct_id', 'why_stopped']].copy()

print(f"Terminated unlabeled trials: {len(terminated_why)}")
print(f"With why_stopped text: {terminated_why['why_stopped'].notna().sum()}")
print(f"Without why_stopped: {terminated_why['why_stopped'].isna().sum()}")

# See the actual reasons
print("\nSample of why_stopped text:")
print(terminated_why['why_stopped'].dropna().head(20).tolist())

Terminated unlabeled trials: 252
With why_stopped text: 235
Without why_stopped: 17

Sample of why_stopped text:
['slow accrual', 'Insufficient funding', 'Due to funding being withdrawn', 'Sponsor decision', 'This study was terminated earlier due to a phase III study that showed this drug inferior to sorafenib', 'Sponsor decision', 'Changing treatment landscape: The availability of nab-paclitaxel with gemcitabine in the second-line setting has changed the feasibility of further recruitment and potential long-term development opportunities of SLC-0111 with gemcitabine alone.', 'Collaborator withdrew support due to a drug supply interruption.', 'Sponsor Decision', 'The study was early terminated following the NIS793 treatment halt and urgent safety measure issued in July 2023, as the continued evaluation of Standard of Care alone will not support the original purpose of this phase 2 clinical trial.', 'Terminated due to safety', 'The total accrual goal of 34 patients was not met. Stage 1 

In [35]:
print("All why_stopped reasons:")
for i, reason in enumerate(terminated_why['why_stopped'].dropna().tolist()):
    print(f"{i+1}: {reason}")

All why_stopped reasons:
1: slow accrual
2: Insufficient funding
3: Due to funding being withdrawn
4: Sponsor decision
5: This study was terminated earlier due to a phase III study that showed this drug inferior to sorafenib
6: Sponsor decision
7: Changing treatment landscape: The availability of nab-paclitaxel with gemcitabine in the second-line setting has changed the feasibility of further recruitment and potential long-term development opportunities of SLC-0111 with gemcitabine alone.
8: Collaborator withdrew support due to a drug supply interruption.
9: Sponsor Decision
10: The study was early terminated following the NIS793 treatment halt and urgent safety measure issued in July 2023, as the continued evaluation of Standard of Care alone will not support the original purpose of this phase 2 clinical trial.
11: Terminated due to safety
12: The total accrual goal of 34 patients was not met. Stage 1 of the study did not meet the interim analysis criteria to move onto Stage 2 of the 

In [ ]:
##This classification was done by feeding into claude the list of "All why stopped reasons"
def classify_why_stopped(text):
    """
    Classify termination reason into:
    FAILURE  = stopped for futility/lack of efficacy → label 0
    EXCLUDE  = stopped for accrual/funding/admin/safety → not a scientific efficacy failure
    """
    if pd.isna(text):
        return 'EXCLUDE'
    
    t = text.lower()
    
    # --- FAILURE: clear efficacy/futility signal ---
    failure_patterns = [
        'futility',
        'lack of efficacy',
        'lack of activity',
        'no efficacy',
        'primary endpoint was not met',
        'did not meet',
        'did not demonstrate',
        'no survival benefit',
        'no differences between',
        'inferior to',
        'insufficient level of efficacy',
        'insufficient efficacy',
        'no objective responses',
        'limited anti-tumor activity',
        'clinical futility',
        'lack of future funding',  # #138 — explicitly combined with futility
        'response rate did not meet',
        'not meet the response criteria',
        'predictive probability was below',
        'not meeting primary endpoint',
        'no longer demonstrates',
        'ineffective',
        'insufficient improvement in os',
        'failed to demonstrate meaningful survival',
        'interim analysis.{0,60}(no|not|insufficient|lack)',
    ]
    
    # --- EXCLUDE: safety stopped ---
    safety_patterns = [
        'toxicity',
        'adverse event',
        'safety',
        'treatment related death',
        'detrimental effect',
        'fluid overload',
        'risk.{0,20}benefit',
    ]
    
    # --- EXCLUDE: accrual/admin/funding ---
    admin_patterns = [
        'accrual',
        'recruitment',
        'enrollment',
        'enroll',
        'funding',
        'sponsor decision',
        'sponsor\'s decision',
        'business',
        'strategic',
        'strategy',
        'pi ',
        'principal investigator',
        'investigator',
        'covid',
        'pandemic',
        'drug supply',
        'manufacturing',
        'company decision',
        'corporate',
        'administrative',
        'logistical',
        'no longer available',
        'no longer needed',
        'obsolete',
        'competing studies',
        'standard of care changed',
        'change in clinical practice',
        'change in strategy',
        'portfolio',
        'grant ended',
        'budget',
        'loss of funding',
        'fda hold',
        'irb',
    ]
    
    import re
    
    is_failure = any(re.search(p, t) for p in failure_patterns)
    is_safety = any(re.search(p, t) for p in safety_patterns)
    is_admin = any(re.search(p, t) for p in admin_patterns)
    
    if is_failure and not is_safety:
        return 'FAILURE'
    else:
        return 'EXCLUDE'

# Apply to terminated unlabeled trials
terminated_unlabeled = unlabeled[unlabeled['overall_status'] == 'TERMINATED'].copy()
terminated_unlabeled['why_stopped_label'] = terminated_unlabeled['why_stopped'].apply(classify_why_stopped)

print("Termination classification:")
print(terminated_unlabeled['why_stopped_label'].value_counts())

# Show the ones labeled FAILURE so we can verify manually
failures_from_terminated = terminated_unlabeled[
    terminated_unlabeled['why_stopped_label'] == 'FAILURE'
][['nct_id', 'why_stopped']].reset_index(drop=True)

print(f"\nTrials labeled FAILURE from termination reason:")
for i, row in failures_from_terminated.iterrows():
    print(f"{i+1}: [{row['nct_id']}] {row['why_stopped']}")

Termination classification:
why_stopped_label
EXCLUDE    228
FAILURE     24
Name: count, dtype: int64

Trials labeled FAILURE from termination reason:
1: [NCT00470535] This study was terminated earlier due to a phase III study that showed this drug inferior to sorafenib
2: [NCT03723915] The total accrual goal of 34 patients was not met. Stage 1 of the study did not meet the interim analysis criteria to move onto Stage 2 of the Simon 2 stage design.
3: [NCT05286827] The study was terminated due to low accrual and insufficient efficacy seen in first few participants treated.
4: [NCT04698915] Futility Analysis
5: [NCT01621243] The study was terminated after a pre-planned futility analyses showed an insufficient level of efficacy in the study population to warrant continuation.
6: [NCT00014651] Teminated by the DSMB because there are no differences between the control group and the Vapreotide group
7: [NCT05116917] Based on the precpecified interim analysis the predictive probability was b

In [37]:
# Build the new failure rows from terminated trials
terminated_failures = terminated_unlabeled[
    terminated_unlabeled['why_stopped_label'] == 'FAILURE'
][['nct_id', 'phase', 'overall_status', 'enrollment', 
   'start_date', 'completion_date', 'why_stopped']].copy()

terminated_failures['label'] = 'FAILURE'
terminated_failures['source'] = 'why_stopped'
terminated_failures['confidence'] = 'HIGH'
terminated_failures['title'] = ''
terminated_failures['abstract'] = terminated_failures['why_stopped']
terminated_failures['pmid'] = None
terminated_failures['primary_endpoint'] = ''
terminated_failures['reasoning'] = terminated_failures['why_stopped']

# Combine with master2
master3 = pd.concat([master2, terminated_failures], ignore_index=True)

# Sanity check — no duplicates
master3 = master3.drop_duplicates(subset='nct_id', keep='first')

print(f"Total labeled trials: {len(master3)}")
print(f"SUCCESS: {(master3['label']=='SUCCESS').sum()}")
print(f"FAILURE: {(master3['label']=='FAILURE').sum()}")
print(f"Success rate: {(master3['label']=='SUCCESS').sum()/len(master3)*100:.1f}%")

print("\nBy phase:")
print(master3.groupby('phase')['label'].apply(
    lambda x: f"{(x=='SUCCESS').sum()}/{len(x)} ({(x=='SUCCESS').mean()*100:.0f}%)"
))

print("\nBy source:")
print(master3.groupby('source')['label'].value_counts())

master3.to_csv('pdac_master_labeled_v3.csv', index=False)
print("\nSaved to pdac_master_labeled_v3.csv")

Total labeled trials: 107
SUCCESS: 47
FAILURE: 60
Success rate: 43.9%

By phase:
phase
PHASE1/PHASE2     6/13 (46%)
PHASE2           27/69 (39%)
PHASE2/PHASE3      2/3 (67%)
PHASE3           12/22 (55%)
Name: label, dtype: object

By source:
source       label  
mistral      FAILURE    30
             SUCCESS    25
regex        SUCCESS    22
             FAILURE     6
why_stopped  FAILURE    24
Name: count, dtype: int64

Saved to pdac_master_labeled_v3.csv
